In [1]:

import glob
from joblib import Parallel, delayed
#from catalog_plotter import plot_var, get_array
import numpy as np
import plotter_config as pc
import sim_name_remap as map
import netCDF4 as nc4
import os
import datetime as dt
from scipy.interpolate import interp1d
import pickle as pkl
import matplotlib.tri as tri
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import datetime
from astropy.time import Time
import utils


In [2]:

def get_simfield(sim, var_strs, date0, date1):

    sim_files = sorted(glob.glob(map.remap[sim]))
    
    sim_files0 = [item for item in sim_files if date0 in item][0]
    find0 = list(sim_files).index(sim_files0)
    sim_files1 = [item for item in sim_files if date1 in item][0]
    find1 = list(sim_files).index(sim_files1)
    
    sim_files = sim_files[find0:find1+1]

    # Crack open the first file and read the convective top information, then loop over the rest
    sim_nc = nc4.Dataset(sim_files[0])
    simlon = sim_nc.variables['lon'][:]
    simlat = sim_nc.variables['lat'][:]
    hyam = sim_nc.variables['hyam'][:]
    hybm = sim_nc.variables['hybm'][:]
    date = sim_nc.variables['date'][:]
    datesec = sim_nc.variables['datesec'][:]
    simfield = sim_nc.variables[var_strs[0]][:]
    if len(var_strs) > 1: 
        for v in range(len(var_strs)-1):
            curr_simfield = sim_nc.variables[var_strs[v+1]][:]
            simfield = simfield + curr_simfield
    
    for f in sim_files[1:]:
        sim_nc = nc4.Dataset(f)        
        date = np.append(date, sim_nc.variables['date'][:])
        datesec = np.append(datesec, sim_nc.variables['datesec'][:])
        
        simfield_f = sim_nc.variables[var_strs[0]][:]
        if len(var_strs) > 1: 
            for v in range(len(var_strs)-1):
                curr_simfield_f = sim_nc.variables[var_strs[v+1]][:]
                simfield_f = simfield_f + curr_simfield_f                
        simfield = np.append(simfield, simfield_f, axis=0)
    
    print('Simulated field shape = ', np.shape(simfield))

    return simfield, simlon, simlat, hyam, hybm, np.asarray(date).flatten(), np.asarray(datesec).flatten()

def interp_to_prslev(var_arr, PS_arr, lon, lat, hyam, hybm, plot_lev):

    var_plev = np.zeros((len(lat), len(lon)))-999

    for ln in range(len(lon)):
        for lt in range(len(lat)):

            curr_PS = PS_arr[lt,ln]
            curr_plev = (hyam*1.0e5 + hybm*curr_PS)/100.  

            plev_f = interp1d(curr_plev, np.squeeze(var_arr[:,lt,ln]), bounds_error=False, fill_value=0)   # We change fill value to zero so topography doesn't mess up the Qconv calculation

            var_plev[lt,ln] = plev_f(plot_lev)
    
    return var_plev
    
def calc_Qconv(simQ_plev, simU_plev, simV_plev, lon, lat):

    Qconv = np.zeros(np.shape(simQ_plev))-999   # Moisture convergence at this pressure level

    # Loop over all lat and lon, do not do the map perimeter so we always have a point before and after the current point
    for ln in range(len(lon)-2):
        for lt in range(len(lat)-2):
    
            curr_deltaU = simU_plev[lt+1,ln+2]-simU_plev[lt+1,ln+0]
            curr_deltalon = 111.2*(lon[ln+2]-lon[ln+0])*np.cos(np.deg2rad(lat[lt+1]))  # In km
            curr_deltaV = simV_plev[lt+2,ln+1]-simV_plev[lt+0,ln+1]
            curr_deltalat = 111.2*(lat[lt+2]-lat[lt+0])   #  In km
    
            Qconv[lt+1,ln+1] = simQ_plev[lt+1,ln+1]*(-(curr_deltaU/curr_deltalon)-(curr_deltaV/curr_deltalat))  
    
    Qconv = Qconv*0.001  # Convert wind to km/s so we can do a cancellation
    Qconv = Qconv*3600.  # Convert from g/kg/s to g/kg/hr
    
    print(np.min(Qconv), np.max(Qconv))

    return Qconv

def plot_map(ax, lon, lat, Qconv, extent, sim, plot_lev, datestr, PSplot, cbar_lev):

    cbar_rgb = plt.get_cmap('coolwarm')

    cf = ax.contourf(lon, lat, Qconv, levels=cbar_lev, cmap=cbar_rgb, extend='both')
    ax.coastlines()
    ax.set_extent(extent)
    ax.set_title(sim)

    ax.contourf(lon, lat, np.squeeze(PSplot)*0.01, levels=[plot_lev/2,plot_lev], colors=['gray','gray'])  # Need to have two, and masking 10 hPa more for extra appearance    

    return cf

    

In [3]:

sims = ['2022_ERA5_ZM_30m','2022_MERRA2_ZM_30m']#,'2022_ERA5_ZMKE_30m','2022_MERRA2_ZMKE_30m']  # These will be used to make a 2x2 plot
#sims = ['2022_ERA5_ZM','2022_MERRA2_ZM','2022_ERA5_ZMKE','2022_MERRA2_ZMKE']  # These will be used to make a 2x2 plot

date0 = '2022-08-18'
date1 = '2022-08-19'
plot_lev = 900.
extent = [105,145,25,45]  # August 19 box, full map boundaries


In [4]:

simQ0, lon, lat, hyam, hybm, date, datesec = get_simfield(sims[0], ['Q'], date0, date1)
simU0, lon, lat, hyam, hybm, date, datesec = get_simfield(sims[0], ['U'], date0, date1)
simV0, lon, lat, hyam, hybm, date, datesec = get_simfield(sims[0], ['V'], date0, date1)
simPS0, lon, lat, hyam, hybm, date, datesec = get_simfield(sims[0], ['PS'], date0, date1)

simQ1, lon, lat, hyam, hybm, date, datesec = get_simfield(sims[1], ['Q'], date0, date1)
simU1, lon, lat, hyam, hybm, date, datesec = get_simfield(sims[1], ['U'], date0, date1)
simV1, lon, lat, hyam, hybm, date, datesec = get_simfield(sims[1], ['V'], date0, date1)
simPS1, lon, lat, hyam, hybm, date, datesec = get_simfield(sims[1], ['PS'], date0, date1)


Simulated field shape =  (54, 32, 192, 288)
Simulated field shape =  (54, 32, 192, 288)
Simulated field shape =  (54, 32, 192, 288)
Simulated field shape =  (54, 192, 288)
Simulated field shape =  (54, 32, 192, 288)
Simulated field shape =  (54, 32, 192, 288)
Simulated field shape =  (54, 32, 192, 288)
Simulated field shape =  (54, 192, 288)


In [5]:

for t in range(np.shape(simQ0)[0]):
#for t in range(5):

    datestr = str(date[t]) + '_' + str(int(datesec[t]/3600.)).zfill(2) + 'Z_' + str(int((datesec[t]/60.) % 60)).zfill(2) + 'min'
    print(datestr)
    
    simQ0_plev = interp_to_prslev(np.squeeze(simQ0[t,:,:,:]), np.squeeze(simPS0[t,:,:]), lon, lat, hyam, hybm, plot_lev)*pc.var_info['Q']['mod']
    simU0_plev = interp_to_prslev(np.squeeze(simU0[t,:,:,:]), np.squeeze(simPS0[t,:,:]), lon, lat, hyam, hybm, plot_lev)
    simV0_plev = interp_to_prslev(np.squeeze(simV0[t,:,:,:]), np.squeeze(simPS0[t,:,:]), lon, lat, hyam, hybm, plot_lev)

    simQ1_plev = interp_to_prslev(np.squeeze(simQ1[t,:,:,:]), np.squeeze(simPS1[t,:,:]), lon, lat, hyam, hybm, plot_lev)*pc.var_info['Q']['mod']
    simU1_plev = interp_to_prslev(np.squeeze(simU1[t,:,:,:]), np.squeeze(simPS1[t,:,:]), lon, lat, hyam, hybm, plot_lev)
    simV1_plev = interp_to_prslev(np.squeeze(simV1[t,:,:,:]), np.squeeze(simPS1[t,:,:]), lon, lat, hyam, hybm, plot_lev)

    Qconv0 = calc_Qconv(simQ0_plev, simU0_plev, simV0_plev, lon, lat)  
    Qconv1 = calc_Qconv(simQ1_plev, simU1_plev, simV1_plev, lon, lat)  

    fig = plt.figure(figsize=(14,10))
    ax0 = fig.add_subplot(1,2,1,projection=ccrs.PlateCarree())
    ax1 = fig.add_subplot(1,2,2,projection=ccrs.PlateCarree())

    cbar_lev = np.arange(-3,3.01,0.5)
    cf = plot_map(ax0, lon, lat, Qconv0, extent, sims[0], plot_lev, datestr, np.squeeze(simPS0[t,:,:]), cbar_lev)
    cf = plot_map(ax1, lon, lat, Qconv1, extent, sims[1], plot_lev, datestr, np.squeeze(simPS1[t,:,:]), cbar_lev)

    cbax = fig.add_axes([0.93,0.3,0.02,0.4])
    cbar = plt.colorbar(cf, cax=cbax, orientation='vertical', ticks = cbar_lev[::4])
    cbar.set_label('Moisture Convergence (g / kg / hr)')
    
    plt.suptitle('Moisture Convergence at ' + str(plot_lev) + ' hPa on ' + datestr)

    plt.savefig('/glade/u/home/wsmith/ASM_Plotter/plots/MoistureConv_' + str(plot_lev) + '_' + datestr + '.png')
    plt.close('all')
    

20220818_00Z_30min
-3596.4 5.635723384316804
-3596.4 4.134655163185173
20220818_01Z_00min
-3596.4 5.854402905615699
-3596.4 4.568234414550516
20220818_01Z_30min
-3596.4 6.19646980100051
-3596.4 4.849516775254876
20220818_02Z_00min
-3596.4 6.4860926968973
-3596.4 4.9558437489756955
20220818_02Z_30min
-3596.4 6.677976316154146
-3596.4 5.2861760359406285
20220818_03Z_00min
-3596.4 6.57804682959217
-3596.4 5.539242275305716
20220818_03Z_30min
-3596.4 6.176113861939198
-3596.4 5.576505805243387
20220818_04Z_00min
-3596.4 5.6329705190279595
-3596.4 5.319404959821759
20220818_04Z_30min
-3596.4 5.503732535585088
-3596.4 4.684370322438354
20220818_05Z_00min
-3596.4 5.245874876003068
-3596.4 4.406484780378966
20220818_05Z_30min
-3596.4 5.230963175776938
-3596.4 4.484632044783555
20220818_06Z_00min
-3596.4 4.960021070092919
-3596.4 4.556030154263973
20220818_06Z_30min
-3596.4 4.703197981520906
-3596.4 4.7310642805665335
20220818_07Z_00min
-3596.4 4.8166307346041535
-3596.4 4.880896370353082
20220